In [ ]:
# ==========================================
# 1. INSTALL NECESSARY TOOLS
# ==========================================
!pip install gdown flask-ngrok pyngrok transformers torch tensorflow joblib Pillow

import os
import gdown
import torch
import numpy as np
import tensorflow as tf
import joblib
import base64
import io
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# ==========================================
# 2. CONFIGURATION & AUTH
# ==========================================
NGROK_TOKEN = "Add Token"
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)

# ==========================================
# 3. DOWNLOAD MODELS FROM DRIVE
# ==========================================
print("📥 Downloading Models from Google Drive...")

# Download Folders (These IDs match your public links)
!gdown --folder 1GnWWFhlsOkCirkmu9O_ElCKL_gSeysIo -O image_models/
!gdown --folder 19Yuhj6IqVrqoQ28TGo-q7CSF-bsb3Vam -O text_models/

print("✅ Download Complete!")

# ==========================================
# 4. LOAD MODELS INTO GPU
# ==========================================
print("🧠 Loading models into T4 GPU...")

# Path to Image Model
IMAGE_MODEL_PATH = 'image_models/Image_Models/symptom_image_model_v2.h5'
image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH)

# Path to Heavy Text Model (DistilBERT v5)
TEXT_MODEL_PATH = 'text_models/heavy_distilbert_ensemble/v5'
text_model = DistilBertForSequenceClassification.from_pretrained(TEXT_MODEL_PATH).to("cuda")
tokenizer = DistilBertTokenizer.from_pretrained(TEXT_MODEL_PATH)

# Path to Label Encoder
LABEL_ENCODER_PATH = 'text_models/heavy_distilbert_ensemble/label_encoder.pkl'
le_text = joblib.load(LABEL_ENCODER_PATH)

print("🚀 All Models Live!")

# ==========================================
# 5. PRE-FLIGHT TEST (PROVE IT WORKS)
# ==========================================
print("\n🔍 --- RUNNING TESTS ---")

# --- Test 1: Text Prediction ---
test_text = "I have a severe headache, high fever, and a stiff neck."
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")
with torch.no_grad():
    outputs = text_model(**inputs)
    idx = torch.argmax(outputs.logits, dim=1).item()
text_result = le_text.inverse_transform([idx])[0]
print(f"✅ Text Model Test: '{test_text}' -> PREDICTION: {text_result}")

# --- Test 2: Image Prediction (Using a dummy image) ---
dummy_img = np.random.rand(1, 224, 224, 3).astype(np.float32)
img_preds = image_model.predict(dummy_img, verbose=0)
classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
img_result = classes[img_preds.argmax()]
print(f"✅ Image Model Test: [Random Pixels] -> PREDICTION: {img_result}")
print("------------------------\n")

# ==========================================
# 6. API ENDPOINTS
# ==========================================

@app.route('/predict-text', methods=['POST'])
def predict_text():
    try:
        data = request.json
        text = data.get('symptoms', '')
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")
        with torch.no_grad():
            outputs = text_model(**inputs)
            prediction = torch.argmax(outputs.logits, dim=1).item()
        result = le_text.inverse_transform([prediction])[0]
        return jsonify({"illness": result, "status": "success"})
    except Exception as e:
        return jsonify({"error": str(e), "status": "error"})

@app.route('/predict-image', methods=['POST'])
def predict_image():
    try:
        data = request.json
        img_b64 = data.get('image', '')
        img_data = base64.b64decode(img_b64)
        img = Image.open(io.BytesIO(img_data)).convert('RGB').resize((224, 224))
        img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
        img_array = tf.expand_dims(img_array, 0)
        
        preds = image_model.predict(img_array)
        classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
        result = classes[preds.argmax()]
        return jsonify({"condition": result, "status": "success"})
    except Exception as e:
        return jsonify({"error": str(e), "status": "error"})

# ==========================================
# 7. START THE TUNNEL
# ==========================================
public_url = ngrok.connect(5000)
print(f"🌍 YOUR PUBLIC API URL IS: {public_url}")
print("\n--- READY TO RECEIVE REQUESTS ---")

app.run(port=5000)

2026-04-18 08:16:22.040663: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776500182.063592     138 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776500182.071037     138 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776500182.090237     138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776500182.090255     138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776500182.090258     138 computation_placer.cc:177] computation placer alr

📥 Downloading Models from Google Drive...
Retrieving folder contents
Processing file 1amZkYch1cEO3zzbCLF84L3esHyMYa9kW symptom_image_model_v1.h5
Processing file 1s3x1iDehjiwgzfAQFqLa7SOkQgyh-Ibt symptom_image_model_v2.h5
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1amZkYch1cEO3zzbCLF84L3esHyMYa9kW
To: /kaggle/working/image_models/Image_Models/symptom_image_model_v1.h5
100%|██████████████████████████████████████| 11.6M/11.6M [00:00<00:00, 46.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1s3x1iDehjiwgzfAQFqLa7SOkQgyh-Ibt
To: /kaggle/working/image_models/Image_Models/symptom_image_model_v2.h5
100%|██████████████████████████████████████| 11.6M/11.6M [00:00<00:00, 72.6MB/s]
Download completed
Retrieving folder contents
Retrieving folder 1VRbxqtQRLRP7bemmLa2je7QfclLiOHdl v1
Processing file 1aY84YmbeRmISJgAOf_FxhTFzS0f4o9gN config.json
Processing file 1rGxMW8VYd9NIjlBv

I0000 00:00:1776500245.482385     138 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776500245.488863     138 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🚀 All Models Live!

🔍 --- RUNNING TESTS ---
✅ Text Model Test: 'I have a severe headache, high fever, and a stiff neck.' -> PREDICTION: Cervical spondylosis


I0000 00:00:1776500249.974219     222 service.cc:152] XLA service 0x7a75e4006da0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776500249.974256     222 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776500249.974261     222 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776500250.617085     222 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-18 08:17:38.230041: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 08:17:38.364930: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1776500259.794535     222 device_co

✅ Image Model Test: [Random Pixels] -> PREDICTION: nv
------------------------

🌍 YOUR PUBLIC API URL IS: NgrokTunnel: "https://rice-elixir-hungrily.ngrok-free.dev" -> "http://localhost:5000"

--- READY TO RECEIVE REQUESTS ---
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:18:22] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:18:23] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:18:33] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:27:15] "GET /predict-text HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:27:29] "GET /predict-image HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:34:58] "POST /predict-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:35:40] "POST /predict-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:37:05] "POST /predict-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 08:38:14] "POST /predict-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Apr/2026 0